In [1]:
from Bio import SeqIO
import os
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

In [2]:
my_record = list(SeqIO.parse("/home/konja/Mini-project-2/ncbi_dataset/gene.fna", "fasta"))

print("Number of records in fasta file:", len(my_record))

Number of records in fasta file: 2


In [3]:
records = my_record[0]
dna_seq = records.seq
print(dna_seq)
print(len(dna_seq))

CCTTAGCAGAGCTGTGGAGTGTGACAATGGTGTTTGTGTCTAAACTATCAAACGCCATTATCACACTAAATAGCTACTGCTAGGC
85


In [4]:
pri_rna = dna_seq.transcribe()
print(pri_rna)

CCUUAGCAGAGCUGUGGAGUGUGACAAUGGUGUUUGUGUCUAAACUAUCAAACGCCAUUAUCACACUAAAUAGCUACUGCUAGGC


In [5]:
import RNA

In [6]:
structure, mfe = RNA.fold(str(pri_rna))
print(structure)
print(mfe)

.((((((((((((((..(((((((.(((((((((((............))))))))))).)))))))..)))))).)))))))).
-45.70000076293945


In [7]:
rna_seq_str = str(pri_rna)
struct_str = structure

In [8]:
# function for map matching base pairs
def get_pair_map(structure):
    pair_map = {}
    stack = []
    for idx, char in enumerate(structure):
        if char == '(':
            stack.append(idx)
        elif char == ')':
            start_idx = stack.pop()
            pair_map[start_idx] = idx
            pair_map[idx] = start_idx
    return pair_map

pair_map = get_pair_map(struct_str)

In [9]:
# Extraction of all 5' stem paired nucleotide positions
five_prime_pairs = [idx for idx, char in enumerate(struct_str) if char == '(']

# cuting 11 base pairs up from the basal junction (Drosha biophysical rule)
drosha_distance = 11

# Checking the number of paired positions
if len(five_prime_pairs) < drosha_distance:
    raise ValueError(f"Structure has only {len(five_prime-pairs)} paired '(' positions," f"cannot cut at distance {drosha_distance}.")

# Calculating the precise 0-indexed positions for the 5' cut with its matching 3' cut
cut_5p = five_prime_pairs[drosha_distance - 1]
cut_3p = pair_map[cut_5p]

# Physical string cleavage stage ; to isolate the pre-miRNA
flanking_5p = rna_seq_str[:cut_5p + 1]
pre_mirna = rna_seq_str[cut_5p + 1 : cut_3p]
flanking_3p = rna_seq_str[cut_3p:]

In [10]:
print("="*40)
print("       DROSHA CLEAVAGE COMPLETION      ")
print("="*40)
print(f"5' Arm Cut Site: Right after index {cut_5p} (Base: {rna_seq_str[cut_5p]})")
print(f"3' Arm Cut Site: Right after index {cut_3p} (Base: {rna_seq_str[cut_3p]})\n")
print(f"Released pre-miRNA Hairpin product ({len(pre_mirna)} nt):")
print(f"--> {pre_mirna}")

       DROSHA CLEAVAGE COMPLETION      
5' Arm Cut Site: Right after index 11 (Base: C)
3' Arm Cut Site: Right after index 72 (Base: G)

Released pre-miRNA Hairpin product (60 nt):
--> UGUGGAGUGUGACAAUGGUGUUUGUGUCUAAACUAUCAAACGCCAUUAUCACACUAAAUA


In [11]:
pre_mirna_seq = "UGUGGAGUGUGACAAUGGUGUUUGUGUCUAAACUAUCAAACGCCAUUAUCACACUAAAUA"
full_structure = structure

# Extracting the exact dot-bracket piece that corresponds to the pre-miRNA
pre_mirna_struct = full_structure[cut_5p + 1 : cut_3p]

In [12]:
print("=== EXPORTIN-5 NUCLEAR EXPORT SIMULATION ===")
print(f"Pre-miRNA Cargo Sequence:  {pre_mirna_seq}")
print(f"Pre-miRNA Cargo Structure: {pre_mirna_struct}\n")

=== EXPORTIN-5 NUCLEAR EXPORT SIMULATION ===
Pre-miRNA Cargo Sequence:  UGUGGAGUGUGACAAUGGUGUUUGUGUCUAAACUAUCAAACGCCAUUAUCACACUAAAUA
Pre-miRNA Cargo Structure: (((..(((((((.(((((((((((............))))))))))).)))))))..)))



In [13]:
# Finding unpaired nucleotides
unpaired_3p_count = 0
for char in reversed(pre_mirna_struct):
    if char == '.':
        unpaired_3p_count += 1
    else:
        break

print(f"Detected 3' Overhang Length: {unpaired_3p_count} nucleotides")

Detected 3' Overhang Length: 0 nucleotides


In [14]:
# Simulating Transport Validation
if unpaired_3p_count == 2:
    print("STATUS: [SUCCESS] Canonical 2-nt 3' overhang detected.")
    print("Exportin-5/RanGTP complex successfully bound to cargo.")
    print("Result: Hairpin successfully translocated through the Nuclear Pore Complex into the Cytoplasm.\n")
else:
    print(f"STATUS: [NON-CANONICAL OVERHANG] Found {unpaired_3p_count}-nt overhang.")
    print("Exportin-5 accommodates minor structural variances in alternative pathways.")
    print("Result: Hairpin forced into Cytoplasmic workspace for processing.\n")

STATUS: [NON-CANONICAL OVERHANG] Found 0-nt overhang.
Exportin-5 accommodates minor structural variances in alternative pathways.
Result: Hairpin forced into Cytoplasmic workspace for processing.



In [15]:
# Defining the cytoplasmic product for Dicer Processing
cytoplasmic_pre_mirna = pre_mirna_seq
print(f"Ready for Dicer processing in the Cytoplasm:")
print(f"--> {cytoplasmic_pre_mirna}")

Ready for Dicer processing in the Cytoplasm:
--> UGUGGAGUGUGACAAUGGUGUUUGUGUCUAAACUAUCAAACGCCAUUAUCACACUAAAUA


In [16]:
pre_mirna_seq = "UGUGGAGUGUGACAAUGGUGUUUGUGUCUAAACUAUCAAACGCCAUUAUCACACUAAAUA"
pre_mirna_struct = "(((..(((((((.(((((((((((............))))))))))).)))))))..)))"

# Re-mapping structural pairings specifically for the pre-miRNA boundaries
def get_pre_pair_map(structure):
    pair_map = {}
    stack = []
    for idx, char in enumerate(structure):
        if char == '(':
            stack.append(idx)
        elif char == ')':
            start_idx = stack.pop()
            pair_map[start_idx] = idx
            pair_map[idx] = start_idx
    return pair_map

pre_pair_map = get_pre_pair_map(pre_mirna_struct)
pre_5p_pairs = [idx for idx, char in enumerate(pre_mirna_struct) if char == '(']

In [17]:
# Applying the dicer physiological rule
dicer_distance = 22

# calculations for functional 5p and 3p mature strands
if len(pre_5p_pairs) < dicer_distance:
    print(f"Error: pre_5p_pairs has only {len(pre_5p_pairs)} paired positions, and cannot use dicer {dicer_distance}.")
else:
    mature_5p_start = 0
    mature_5p_end = pre_5p_pairs[dicer_distance - 1]
    mature_3p_start = pre_pair_map[mature_5p_end]
    mature_3p_end = len(pre_mirna_seq)
    # Slicing the sequence for isolation of functional active products
    mature_5p = pre_mirna_seq[mature_5p_start : mature_5p_end + 1] 
    mature_3p = pre_mirna_seq[mature_3p_start : mature_3p_end]
    terminal_loop_product = pre_mirna_seq[mature_5p_end + 1 : mature_3p_start]

Error: pre_5p_pairs has only 21 paired positions, and cannot use dicer 22.


In [18]:
# Changing dicer
dicer_distance = 21

mature_5p_start = 0
mature_5p_end = pre_5p_pairs[dicer_distance - 1]

mature_3p_start = pre_pair_map[mature_5p_end]
mature_3p_end = len(pre_mirna_seq)

mature_5p = pre_mirna_seq[mature_5p_start : mature_5p_end + 1]
mature_3p = pre_mirna_seq[mature_3p_start : mature_3p_end]
terminal_loop_product = pre_mirna_seq[mature_5p_end + 1 : mature_3p_start]

In [19]:
print("="*45)
print("         DICER CYTOPLASMIC PROCESSING        ")
print("="*45)
print(f"Mature 5p miRNA Product ({len(mature_5p)} nt):  {mature_5p}")
print(f"Released Terminal Loop  ({len(terminal_loop_product)} nt):  {terminal_loop_product}")
print(f"Mature 3p miRNA Product ({len(mature_3p)} nt):  {mature_3p}\n")
print("Status: [SUCCESS] Functional miRNA duplex generated.")
print("Ready for RISC complex assembly and target mRNA silencer strand selection.")

         DICER CYTOPLASMIC PROCESSING        
Mature 5p miRNA Product (24 nt):  UGUGGAGUGUGACAAUGGUGUUUG
Released Terminal Loop  (12 nt):  UGUCUAAACUAU
Mature 3p miRNA Product (24 nt):  CAAACGCCAUUAUCACACUAAAUA

Status: [SUCCESS] Functional miRNA duplex generated.
Ready for RISC complex assembly and target mRNA silencer strand selection.


In [20]:
my_5p = "UGUGGAGUGUGACAAUGGUGUUUG"
my_3p = "CAAACGCCAUUAUCACACUAAAUA"

# Displaying the duplex configuration
print(f"5' Arm (5p): 5' - {my_5p} - 3'")
print(f"                  ||||||||||||||||||||||||")
print(f"3' Arm (3p): 3' - {my_3p[::-1]} - 5'\n")

5' Arm (5p): 5' - UGUGGAGUGUGACAAUGGUGUUUG - 3'
                  ||||||||||||||||||||||||
3' Arm (3p): 3' - AUAAAUCACACUAUUACCGCAAAC - 5'



In [21]:
# Applying the thermodynamic asymmetry rule
five_prime_5p_end = my_5p[:4]
five_prime_3p_end = my_3p[:4]

gc_5p = five_prime_5p_end.count('G') + five_prime_5p_end.count('C')
gc_3p = five_prime_3p_end.count('G') + five_prime_3p_end.count('C')

In [22]:
print(f"Thermodynamic profiling (First 4 nt at 5' ends):")
print(f" -> 5p terminus: {five_prime_5p_end} (G/C count = {gc_5p})")
print(f" -> 3p terminus: {five_prime_3p_end} (G/C count = {gc_3p})\n")

Thermodynamic profiling (First 4 nt at 5' ends):
 -> 5p terminus: UGUG (G/C count = 2)
 -> 3p terminus: CAAA (G/C count = 1)



In [23]:
# Applying the selection logic
if gc_5p < gc_3p:
    guide_strand = my_5p
    passenger_strand = my_3p
    selection_verdict = "5p strand selected as GUIDE due to lower 5' end stability."
elif gc_3p < gc_5p:
    guide_strand = my_3p
    passenger_strand = my_5p
    selection_verdict = "3p strand selected as GUIDE due to lower 5' end stability."
else:
    # If tie, default to canonical ensemble assumption or dual-strand selection
    guide_strand = my_5p
    passenger_strand = my_3p
    selection_verdict = "Strands have symmetric 5' stability. Both may act as guides (Co-dominance)."

In [24]:
print(f"RISC Assembly Verdict:")
print(f"STATUS: {selection_verdict}")
print(f"Active Functional Guide miRNA: 5' - {guide_strand} - 3'")
print(f"Discarded Passenger Strand:    5' - {passenger_strand} - 3'")

RISC Assembly Verdict:
STATUS: 3p strand selected as GUIDE due to lower 5' end stability.
Active Functional Guide miRNA: 5' - CAAACGCCAUUAUCACACUAAAUA - 3'
Discarded Passenger Strand:    5' - UGUGGAGUGUGACAAUGGUGUUUG - 3'


In [25]:
# Saving mature miRNA to FASTA
file_name = "mature_mirna.fasta"

# Defining sequence and fasta header
fasta_header = ">m_miR_3p Mature_3p_guide_miRNA | Seed: AAACGCC | Pre-AGO2_Loading\n"
fasta_sequence = "CAAACGCCAUUAUCACACUAAAUA\n"

with open(file_name, "w") as f:
    f.write(fasta_header)
    f.write(fasta_sequence)

print(f"SUCCESS! Created file: {file_name}")

SUCCESS! Created file: mature_mirna.fasta


In [26]:
mature_mirna = "CAAACGCCAUUAUCACACUAAAUA"

# Maping out the domain anchors within AGO2
domain_5p_mid = mature_mirna[0]   # The anchoring 5' nucleotide position
domain_3p_paz = mature_mirna[-1]  # The anchoring 3' nucleotide position

# 3. Extract the critical Seed Region (Nucleotides 2 to 8)
seed_region = mature_mirna[1:8]

In [27]:
print(f"Structural Domain Allocations inside RISC:")
print(f" -> 5' MID Domain Anchor base: {domain_5p_mid}")
print(f" -> 3' PAZ Domain Anchor base: {domain_3p_paz}")
print(f" -> Exposed Regulatory SEED Region (nt 2-8): **{seed_region}**\n")

print("Status: [SUCCESS] RISC loading complete.")

Structural Domain Allocations inside RISC:
 -> 5' MID Domain Anchor base: C
 -> 3' PAZ Domain Anchor base: A
 -> Exposed Regulatory SEED Region (nt 2-8): **AAACGCC**

Status: [SUCCESS] RISC loading complete.


In [28]:
# Exporting domain allocations to FASTA
file_name = "AGO2_loaded_guide.fasta"

fasta_header = ">m_miR_3p_AGO2_loaded Guide_miRNA | Seed: AAACGCC | Status: Active_RISC\n"
fasta_sequence = "CAAACGCCAUUAUCACACUAAAUA\n"

with open(file_name, "w") as f:
    f.write(fasta_header)
    f.write(fasta_sequence)

print(f"SUCCESS! Created file: {file_name}")

SUCCESS! Created file: AGO2_loaded_guide.fasta
